In [ ]:
import asyncio
import pandas as pd
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_score, extract_string
from vpei.common_variables import *
from vpei.epistemic_consistency.experiment_utils import *
from vpei.epistemic_consistency.experiment_types import carry_out_comparative_experiment_with_ground_truth
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS

input_file = "./data/AVeriTeC_dataset_train_preprocessed.csv"
df = pd.read_csv(input_file)
df.rename(columns={"claim":"statement"}, inplace=True) # to make clearer prompt templates

df_correct = df[(df["factual_accuracy"]==True)]
df_incorrect = df[(df["factual_accuracy"]==False)]
df

In [ ]:
experiment_name = "factual_vs_false_statement_detection"
system_prompt = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth"]["system_prompt"]
user_prompt_template = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth"]["user_prompt_template"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template)

In [ ]:
# model_name = "gpt-4o-mini"
# model_name = "gpt-5"
model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})
name_1 = "P.P."
name_2 = "A.J."
political_attitude_1="conservative"
political_attitude_2="progressive"
statement_1 = df_correct.iloc[0]['statement']
statement_2 = df_incorrect.iloc[1]['statement']
user_prompt = user_prompt_template.format(name_1=name_1, political_attitude_1=political_attitude_1, statement_1=statement_1, name_2=name_2, 
                                          political_attitude_2=political_attitude_2, statement_2=statement_2
                                          )

messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
make_llm_request(model_name, messages, **model_kwargs)

In [ ]:
models = ["gpt-5-mini"]

n = 1
custom_model_kwargs = {}
stimuli_factors = ["statement"]
additional_variables_from_df_to_save = [] 
path_to_save_model_outputs = "./comparative_experiment_with_ground_truth"
random_seed = 42

In [ ]:
payloads = await carry_out_comparative_experiment_with_ground_truth(models=models, df_correct=df_correct, df_incorrect=df_incorrect, n=n, system_prompt=system_prompt, 
                                                                    user_prompt_template=user_prompt_template, stimuli_factors=stimuli_factors, 
                                                                    additional_variables_from_df_to_save=additional_variables_from_df_to_save,
                                                                    custom_model_kwargs=custom_model_kwargs, path_to_save_model_outputs=path_to_save_model_outputs, 
                                                                    random_seed=random_seed)

print_comparative_experiment_results(payloads, models)